# BasaQuest — Engagement Classifier (Surface vs. Deep Learner)

Trains the **logistic regression** the manuscript specifies for the History
module: from a pupil's logged in-game behavior, classify them as a **Surface**
or **Deep** learner and recommend post-arc content accordingly.

**Objective (from the study):** at least **80% accuracy** against teacher-assessed
ground truth.

**How this fits the app:** a logistic regression is just weights. You train here,
export the coefficients, and paste them into `src/game/classifier.ts` — inference
runs in the browser as `sigmoid(w · x + b)`, no ML server needed.

**Workflow**
1. Define the feature set (must match `FEATURE_NAMES` in `classifier.ts`).
2. Load data. *Now:* synthetic data so the pipeline runs. *Later:* real pilot
   sessions (logged events + teacher labels).
3. Train + evaluate (accuracy, cross-validation, confusion matrix).
4. Export weights ready to paste into `classifier.ts`.


## 1. Setup


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# MUST match FEATURE_NAMES (and their order) in src/game/classifier.ts
FEATURE_NAMES = [
    "avg_decision_ms",           # slower, deliberate decisions -> deeper engagement
    "timeout_rate",              # fraction of decisions that timed out -> disengagement
    "avg_minigame_attempts",     # more retries -> persistence
    "minigame_completion_rate",  # finished mini-games -> engagement
]
print("Features:", FEATURE_NAMES)

## 2. Data

### 2a. Synthetic data (use now)

This generates plausible sessions for two latent groups so the whole pipeline
runs and is testable **before** the pilot. The overlap between groups is
deliberate — real behavior is noisy, so accuracy lands in a realistic ~85–92%
range rather than a fake 100%.

> Replace this with **2b** once you have real pilot sessions.


In [ ]:
def make_synthetic(n_per_class=150, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)

    # Deep learners (label 1): deliberate decisions, rarely time out,
    # persistent on mini-games, high completion. Distributions overlap the
    # surface group on purpose so accuracy is realistic, not a leaky 100%.
    deep = pd.DataFrame({
        "avg_decision_ms":          rng.normal(4200, 1400, n_per_class).clip(400, 8000),
        "timeout_rate":             rng.beta(1.3, 10, n_per_class),          # mostly low
        "avg_minigame_attempts":    rng.normal(2.0, 0.65, n_per_class).clip(1, 5),
        "minigame_completion_rate": rng.uniform(0.63, 1.0, n_per_class),
    })
    deep["label"] = 1

    # Surface learners (label 0): rush or stall, more timeouts, give up sooner,
    # lower completion.
    surface = pd.DataFrame({
        "avg_decision_ms":          rng.normal(2400, 1200, n_per_class).clip(300, 8000),
        "timeout_rate":             rng.beta(2.2, 6, n_per_class),           # spread higher
        "avg_minigame_attempts":    rng.normal(1.35, 0.55, n_per_class).clip(1, 5),
        "minigame_completion_rate": rng.uniform(0.38, 0.86, n_per_class),
    })
    surface["label"] = 0

    df = pd.concat([deep, surface], ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

df = make_synthetic()
print(df.shape)
df.groupby("label")[FEATURE_NAMES].mean()

### 2b. Real pilot data (use later)

When the pilot is done, export the behavior logs from Supabase
(`history_behavior_logs`) and attach each session's teacher label. The helper
below turns raw events into the same feature row the app computes in
`classifier.ts` — keep the two in sync.

Two inputs:
- `events_df`: columns `session_id, type, node_id, payload` (payload = dict).
- `labels_df`: columns `session_id, label` (1 = deep, 0 = surface) from the teacher.


In [ ]:
def features_from_events(events_df: pd.DataFrame) -> pd.DataFrame:
    """Mirror of extractFeatures() in classifier.ts, per session."""
    rows = []
    for sid, g in events_df.groupby("session_id"):
        dec = g[g["type"] == "decision_made"]
        starts = g[g["type"] == "minigame_start"]
        comps = g[g["type"] == "minigame_complete"]

        dec_ms = [float(p.get("msElapsed", 0)) for p in dec["payload"]]
        timeouts = sum(1 for p in dec["payload"] if p.get("timedOut") is True)
        attempts = [float(p.get("attempts", 1)) for p in comps["payload"]]

        rows.append({
            "session_id": sid,
            "avg_decision_ms": np.mean(dec_ms) if dec_ms else 0.0,
            "timeout_rate": (timeouts / len(dec)) if len(dec) else 0.0,
            "avg_minigame_attempts": np.mean(attempts) if attempts else 0.0,
            "minigame_completion_rate": (len(comps) / len(starts)) if len(starts) else 0.0,
        })
    return pd.DataFrame(rows)

# --- Uncomment when you have real CSVs ---
# events_df = pd.read_json("pilot_events.json")      # or read_csv + json-parse payload
# labels_df = pd.read_csv("teacher_labels.csv")
# feats = features_from_events(events_df)
# df = feats.merge(labels_df, on="session_id")[FEATURE_NAMES + ["label"]]
# print(df.shape)

## 3. Train + evaluate


In [ ]:
X = df[FEATURE_NAMES].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# StandardScaler matters here: avg_decision_ms is ~thousands while the rates are
# 0..1. We fold the scaler back into raw-space weights before exporting (step 4),
# so the browser can apply the weights directly to raw features.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cv = cross_val_score(model, X, y, cv=5)

print(f"Test accuracy: {acc:.3f}")
print(f"5-fold CV accuracy: {cv.mean():.3f} +/- {cv.std():.3f}")
print(f"Meets >=80% objective: {acc >= 0.80}")
print()
print(classification_report(y_test, y_pred, target_names=["surface", "deep"]))
print("Confusion matrix [rows=true, cols=pred] (surface, deep):")
print(confusion_matrix(y_test, y_pred))

## 4. Export weights for `classifier.ts`

The app computes `z = bias + Σ weight_i · (x_i / scale_i)` on **raw** features.
We trained on **standardized** features, so we fold the scaler
`(x - mean)/std` into the coefficients:

- `weight_i = coef_i / std_i`
- `bias = intercept - Σ (coef_i · mean_i / std_i)`
- `scale_i = 1` (folding is already done)

Then the exported numbers apply directly to the raw feature vector.


In [ ]:
scaler = model.named_steps["standardscaler"]
clf = model.named_steps["logisticregression"]

mean = scaler.mean_
std = scaler.scale_
coef = clf.coef_[0]
intercept = clf.intercept_[0]

folded_weights = coef / std
folded_bias = intercept - np.sum(coef * mean / std)

# Sanity check: raw-space math must equal the sklearn pipeline's probabilities.
def sigmoid(z): return 1 / (1 + np.exp(-z))
z_raw = folded_bias + X_test @ folded_weights
p_raw = sigmoid(z_raw)
p_sk = model.predict_proba(X_test)[:, 1]
print("Max abs diff (raw-fold vs sklearn):", np.max(np.abs(p_raw - p_sk)))  # ~1e-12

### Paste-ready TypeScript


In [ ]:
def fmt(arr): return "[" + ", ".join(f"{v:.6g}" for v in arr) + "]"

print("// --- Generated by engagement_classifier.ipynb ---")
print(f"// Trained on {len(df)} sessions | test acc {acc:.3f} | CV {cv.mean():.3f}")
print("// Feature order:", FEATURE_NAMES)
print(f"const PLACEHOLDER_WEIGHTS = {fmt(folded_weights)};")
print(f"const PLACEHOLDER_BIAS = {folded_bias:.6g};")
print(f"const FEATURE_SCALE = {fmt([1, 1, 1, 1])};")

Copy the three lines above into `src/game/classifier.ts` (replacing the
placeholders). Rename the constants if you like — just keep the values and order.


### Also save `weights.json` (optional)


In [ ]:
import json
weights = {
    "feature_names": FEATURE_NAMES,
    "weights": [float(w) for w in folded_weights],
    "bias": float(folded_bias),
    "scale": [1, 1, 1, 1],
    "trained_on_sessions": int(len(df)),
    "test_accuracy": float(acc),
    "cv_accuracy_mean": float(cv.mean()),
}
with open("weights.json", "w") as f:
    json.dump(weights, f, indent=2)
print(json.dumps(weights, indent=2))
# In Colab: from google.colab import files; files.download("weights.json")